# Action Distribution Plots

Lean notebook backed by `action_distribution_metrics.py`. The original large notebook remains at `../action_distribution_plots.ipynb`.


In [13]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    local_metrics = candidate / "action_distribution_metrics.py"
    repo_metrics = candidate / "Topology_Task" / "configs" / "zz_print_metrics" / "action_distribution_metrics.py"
    if local_metrics.exists():
        sys.path.insert(0, str(candidate))
        break
    if repo_metrics.exists():
        sys.path.insert(0, str(repo_metrics.parent))
        break

import action_distribution_metrics as adm
adm = importlib.reload(adm)
import wandb_metrics as wm
wm = importlib.reload(wm)
print("action_distribution_metrics:", adm.__file__)
print("wandb_metrics:", wm.__file__)


action_distribution_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/zz_print_metrics/action_distribution_metrics.py
wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/zz_print_metrics/wandb_metrics.py


## Choose Runs To Load Or Download


In [14]:
import importlib
wm = importlib.reload(wm)

# Pick config folder(s) under Topology_Task/configs, or use None / "None" / "all" for every run.
# Examples:
# CONFIG_FOLDERS_TO_DOWNLOAD = "intervention_gate_15"
# CONFIG_FOLDERS_TO_DOWNLOAD = ["intervention_gate_15", "phase4_sparse_control_16"]
# CONFIG_FOLDERS_TO_DOWNLOAD = "intervention_gate_15,phase4_sparse_control_16"
# CONFIG_FOLDERS_TO_DOWNLOAD = "all"
CONFIG_FOLDERS_TO_DOWNLOAD = [
    "intervention_gate_15",
    "phase4_sparse_control_16",
    "heuristic_vs_gate_s0_s1_s2",
    "adaptive_intervention_budget_7",
]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = False

# Useful when W&B has newer data than the local cache, especially for old scan_history fallback caches.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = True

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS_TO_DOWNLOAD)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)


Config folder filter:
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/intervention_gate_15
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/phase4_sparse_control_16
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/heuristic_vs_gate_s0_s1_s2
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/adaptive_intervention_budget_7
Matched config run-name candidates: 69
RUN_NAME_REGEX = ^\s*(?:ig_00_phase2_base_s0|ig_00_phase2_base_s1|ig_00_phase2_base_s2|ig_01_entropy_decay_s0|ig_01_entropy_decay_s1|ig_01_entropy_decay_s2|ig_02_topo001_entropy_decay_s0|ig_02_topo001_entropy_decay_s1|ig_02_topo001_entropy_decay_s2|ig_03_topo005_entropy_decay_s0|ig_03_topo005_entropy_decay_s1|ig_03_topo005_entropy_decay_s2|ig_04_topo010_entropy_decay_s0|ig_04_topo010_entropy_decay_s1|ig_04_topo010_entropy_decay_s2|sparse16_flat_p000_s0|sparse16_flat_p000_s1|sparse16_flat_p000_s2|sparse16_flat_p001_s0|sparse16_flat_p001_s1|sparse

## Load Or Download Selected W&B Histories


In [15]:
download_data = wm.load_wandb_data()
downloaded_runs_df = download_data.runs_df
print(f"Matching W&B runs: {len(downloaded_runs_df)}")


Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 69 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    38
running     16
crashed     15
History artifact setup: local_only=True, runs_df=69
[ 1/69] loading artifact cache: aib_00_flat_local_t020_s0 
    loaded 278 rows, 140 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/aib_00_flat_local_t020_s0__MAPPO_bus14_T_0_0__I__1781866679_28407/history.parquet in 0.1s
[ 2/69] loading artifact cache: aib_00_flat_local_t020_s1
    loaded 328 rows, 140 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/aib_00_flat_lo

## Load Cached Histories And Shared Tables


In [16]:
# Use final evaluation action metrics for the action-distribution plots.
# Set ACTION_METRIC_SOURCE = "train" to reproduce the old rollout-training plots.
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"

# Final-window plots average the last logged points at or before this step.
# Use None to treat each run's actual last logged point as final.
FINAL_SUMMARY_STEP_M = 11.0  # Set to None to use each run's actual last logged point.

ctx = adm.load_action_distribution_context(
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    final_summary_step_m=FINAL_SUMMARY_STEP_M,
    save_figures=True,
    show_figures=False,
)


Expected configs: 69
Cached expected runs: 69 / 69
[  1/69] loading aib_00_flat_local_t020_s0
[  2/69] loading aib_00_flat_local_t020_s1
[  3/69] loading aib_00_flat_local_t020_s2
[  4/69] loading aib_01_flat_local_t010_s0
[  5/69] loading aib_01_flat_local_t010_s1
[  6/69] loading aib_01_flat_local_t010_s2
[  7/69] loading aib_02_flat_local_t035_s0
[  8/69] loading aib_02_flat_local_t035_s1
[  9/69] loading aib_02_flat_local_t035_s2
[ 10/69] loading aib_03_gate_hgreedy_sep_local_t020_s0
[ 11/69] loading aib_03_gate_hgreedy_sep_local_t020_s1
[ 12/69] loading aib_03_gate_hgreedy_sep_local_t020_s2
[ 13/69] loading aib_04_flat_nonidle_t020_s0
[ 14/69] loading aib_04_flat_nonidle_t020_s1
[ 15/69] loading aib_04_flat_nonidle_t020_s2
[ 16/69] loading hvg_00_baseline_s0
[ 17/69] loading hvg_00_baseline_s1
[ 18/69] loading hvg_00_baseline_s2
[ 19/69] loading hvg_01_eval_rho090_s0
[ 20/69] loading hvg_01_eval_rho090_s1
[ 21/69] loading hvg_01_eval_rho090_s2
[ 22/69] loading hvg_02_gate_final_ma

## Last-5logged action 0 fraction by agent for all runs


In [17]:
last5_action0 = adm.last5_logged_action0_fraction_by_agent_all_runs(ctx)
adm.display_metric_figures(last5_action0)


No final action-0 data for intervention_gate_15
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_phase4_sparse_control_16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_heuristic_vs_gate_s0_s1_s2.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_adaptive_intervention_budget_7.html


## Seed-Aggregated Last-5-Logged Action-0 Fraction By Agent


In [18]:
seed_action0 = adm.seed_aggregated_last5_logged_action0_fraction_by_agent(ctx)
adm.display_metric_figures(seed_action0)


No seed-aggregated final action-0 data for intervention_gate_15
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_phase4_sparse_control_16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_heuristic_vs_gate_s0_s1_s2.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_adaptive_intervention_budget_7.html


## Survival vs Action-0 / Non-Idle Over Time


In [19]:
survival_action = adm.survival_vs_action0_non_idle_over_time(ctx)
adm.display_metric_figures(survival_action)


Missing survival or action-behavior data for intervention_gate_15
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_phase4_sparse_control_16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_heuristic_vs_gate_s0_s1_s2.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_adaptive_intervention_budget_7.html


## Action-0 / Survival Tradeoff


In [20]:
action0_survival = adm.action0_survival_tradeoff(ctx)
adm.display_metric_figures(action0_survival)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_adaptive_intervention_budget_7.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_heuristic_vs_gate_s0_s1_s2.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_phase4_sparse_control_16.html


## Entropy Collapse vs Action-0 Confidence


In [21]:
entropy_action0 = adm.entropy_collapse_vs_action0_confidence(ctx)
adm.display_metric_figures(entropy_action0)


No evaluation entropy metrics found; skipping entropy/action-0 plots.


## Agent Non-Idle Imbalance


In [22]:
agent_imbalance = adm.agent_non_idle_imbalance(ctx)
adm.display_metric_figures(agent_imbalance)


No agent imbalance time-series data for intervention_gate_15
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_phase4_sparse_control_16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_heuristic_vs_gate_s0_s1_s2.html
No final agent imbalance data for intervention_gate_15
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_agent_non_idle_imbalance_phase4_sparse_control_16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_agent_non_idle_imbalance_heuristic_vs_gate_s0_s1_s2.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_adaptive_intervention_budget_7

## Joint Action Coordination


In [23]:
joint_coordination = adm.joint_action_coordination(ctx)
adm.display_metric_figures(joint_coordination)


No scalar evaluation joint-action metrics found; skipping joint coordination plots. Eval joint coordination needs eval non_idle_agents_count_* scalars or trace-table processing.


## Gate Probability vs Actual Intervention Fraction


In [24]:
gate_calibration = adm.gate_probability_vs_actual_intervention_fraction(ctx)
adm.display_metric_figures(gate_calibration)


No evaluation gate-probability metrics found; skipping gate probability vs actual plots. Eval logs final gate intervention fractions, but not gate probabilities.
